In [3]:
import pandas as pd

df = pd.read_csv("careflow_synthetic_500_patients.csv")

df["event_order"] = pd.to_numeric(
    df["event_order"], errors="coerce"
)

df = df.sort_values(["case_id", "event_order"])

journeys = df.groupby("case_id")["activity"].apply(list)

path_counts = journeys.apply(
    lambda x: " → ".join(x)
).value_counts()

print(path_counts)

activity
Registration → Triage → Doctor Consultation → Lab Test → Treatment → Discharge                                                           169
Registration → Triage → Doctor Consultation → Treatment → Discharge                                                                       70
Registration → Triage → Doctor Consultation → Lab Test → Imaging → Treatment → Discharge                                                  55
Registration → Triage → Doctor Consultation → Lab Test → Imaging → Doctor Consultation → Lab Test → Treatment → Discharge                 50
Registration → Triage → Doctor Consultation → Lab Test → Treatment → Discharge → Follow-up                                                35
Registration → Triage → Doctor Consultation → Lab Test → Doctor Consultation → Lab Test → Treatment → Discharge                           33
Registration → Triage → Doctor Consultation → Imaging → Treatment → Discharge                                                             26
Regi

In [5]:
unusual_transitions = [
    ("Lab Test", "Doctor Consultation"),
    ("Imaging", "Doctor Consultation"),
    ("Doctor Consultation", "Imaging"),
    ("Doctor Consultation", "Treatment")
]

for current, next_act in unusual_transitions:

    matching = df[
        (df["activity"] == current) &
        (df["next_activity"] == next_act)
    ]

    print("\n--------------------------------")
    print(current, "->", next_act)
    print("Transition count:", len(matching))
    print("Patient cases:", matching["case_id"].nunique())

    print("Sample patient IDs:")
    print(matching["case_id"].unique()[:10])


--------------------------------
Lab Test -> Doctor Consultation
Transition count: 48
Patient cases: 48
Sample patient IDs:
<StringArray>
['P0009', 'P0012', 'P0013', 'P0016', 'P0044', 'P0051', 'P0059', 'P0062',
 'P0073', 'P0086']
Length: 10, dtype: str

--------------------------------
Imaging -> Doctor Consultation
Transition count: 61
Patient cases: 61
Sample patient IDs:
<StringArray>
['P0003', 'P0023', 'P0035', 'P0036', 'P0039', 'P0041', 'P0056', 'P0090',
 'P0093', 'P0096']
Length: 10, dtype: str

--------------------------------
Doctor Consultation -> Imaging
Transition count: 30
Patient cases: 30
Sample patient IDs:
<StringArray>
['P0010', 'P0025', 'P0040', 'P0046', 'P0058', 'P0075', 'P0087', 'P0114',
 'P0164', 'P0177']
Length: 10, dtype: str

--------------------------------
Doctor Consultation -> Treatment
Transition count: 91
Patient cases: 91
Sample patient IDs:
<StringArray>
['P0004', 'P0018', 'P0019', 'P0026', 'P0028', 'P0029', 'P0034', 'P0037',
 'P0045', 'P0047']
Length: 

In [6]:
for current, next_act in unusual_transitions:

    matching = df[
        (df["activity"] == current) &
        (df["next_activity"] == next_act)
    ]

    print("\nTRANSITION:", current, "->", next_act)

    sample_ids = matching["case_id"].drop_duplicates().head(3)

    for case in sample_ids:

        patient = df[df["case_id"] == case].sort_values(
            "event_order"
        )

        print("\nPatient:", case)

        print(patient[
            ["event_order", "activity",
             "timestamp", "journey_type", "is_loop_back"]
        ].to_string(index=False))


TRANSITION: Lab Test -> Doctor Consultation

Patient: P0009
 event_order            activity           timestamp journey_type  is_loop_back
           1        Registration 2026-09-08 14:45:00    loop_back         False
           2              Triage 2026-09-08 14:54:00    loop_back         False
           3 Doctor Consultation 2026-09-08 15:40:00    loop_back          True
           4            Lab Test 2026-09-08 17:01:00    loop_back         False
           5 Doctor Consultation 2026-09-08 18:24:00    loop_back          True
           6            Lab Test 2026-09-08 19:07:00    loop_back         False
           7           Treatment 2026-09-08 21:02:00    loop_back         False
           8           Discharge 2026-09-08 21:54:00    loop_back         False
           9           Follow-up 2026-09-08 22:12:00    loop_back         False

Patient: P0012
 event_order            activity           timestamp journey_type  is_loop_back
           1        Registration 2026-09-09